# Exercise 11 — Inspect and Use a Small Language Model

Use the same pretrained model as Tutorial 11. There is no training and no paid API. The preparation code is complete; focus on inspecting tokens, next-token probabilities, decoding settings and prompt context.

## Open this notebook

- **Colab:** run the setup cell first. It clones the public course repository and changes into `11 Large Language Models`.
- **Your own computer:** open the notebook from this lesson folder.

The model is downloaded once per new environment and reused by all questions.

In [ ]:
from pathlib import Path
import os

def in_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if in_colab():
    print("Running in Google Colab")
    REPO_URL = "https://github.com/hku-kejintao/AI-Code.git"
    LESSON_DIR = "11 Large Language Models"
    base = Path("/content")
    repo = base / "AI-Code"
    os.chdir(base)
    if not repo.exists():
        !git clone {REPO_URL}
    lesson_path = repo / LESSON_DIR
    if not lesson_path.exists():
        raise FileNotFoundError(f"Lesson folder not found: {lesson_path}")
    os.chdir(lesson_path)
else:
    print("Running on your own computer")

print("Current folder:", os.getcwd())

## Check packages and load the model

The installation cell runs when Transformers is missing or its version is incompatible. Loading may take a minute on the first run because the public model weights must be downloaded.

In [ ]:
from importlib.metadata import PackageNotFoundError, version
from packaging.requirements import Requirement
import subprocess
import sys

required_packages = {
    "transformers": "transformers>=4.57,<5",
}

install_requirements = []
for package, requirement_text in required_packages.items():
    requirement = Requirement(requirement_text)
    try:
        installed_version = version(package)
    except PackageNotFoundError:
        install_requirements.append(requirement_text)
        continue
    if not requirement.specifier.contains(installed_version, prereleases=True):
        install_requirements.append(requirement_text)

if install_requirements:
    print("Installing or updating:", install_requirements)
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        *install_requirements,
    ])
else:
    print("Required packages are already installed.")

import numpy as np
import pandas as pd
pd.set_option("display.max_colwidth", None)
import matplotlib.pyplot as plt
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

print("torch", torch.__version__)
print("transformers", transformers.__version__)
print("ready")

In [ ]:
MODEL_NAME = "HuggingFaceTB/SmolLM2-360M-Instruct"

if torch.cuda.is_available():
    device = torch.device("cuda")
    model_dtype = torch.float16
elif torch.backends.mps.is_available():
    device = torch.device("mps")
    model_dtype = torch.float16
else:
    device = torch.device("cpu")
    model_dtype = torch.float16

print("Loading", MODEL_NAME)
print("Device:", device)
print("Model dtype:", model_dtype)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=model_dtype,
)
model.to(device)
model.eval()

model.generation_config.do_sample = False
model.generation_config.temperature = None
model.generation_config.top_p = None
model.generation_config.top_k = None

if tokenizer.pad_token_id is None:
    tokenizer.pad_token_id = tokenizer.eos_token_id

parameter_count = sum(parameter.numel() for parameter in model.parameters())
print(f"Parameters: {parameter_count / 1e6:,.1f} million")
print("Vocabulary size:", model.config.vocab_size)
print("Transformer layers:", model.config.num_hidden_layers)
print("No training will be performed: model.eval() and torch.inference_mode() are used.")

In [ ]:
COLORS = ["#4C78A8", "#F58518", "#54A24B", "#B279A2"]
plt.rcParams.update({
    "figure.dpi": 110,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titleweight": "bold",
})

## Prepared helper functions

These functions format chat messages, generate a short response and extract the most likely next tokens. Read their input and output variables before answering the questions.

In [ ]:
def prepare_chat_inputs(user_message, system_message="Answer clearly and briefly."):
    messages = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]
    formatted_text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(formatted_text, return_tensors="pt")
    model_inputs = {name: value.to(device) for name, value in model_inputs.items()}
    return model_inputs, formatted_text


def generate_reply(
    user_message,
    system_message="Answer clearly and briefly.",
    max_new_tokens=60,
    do_sample=False,
    temperature=0.8,
    top_p=0.9,
    seed=42,
):
    model_inputs, _ = prepare_chat_inputs(user_message, system_message)
    generation_options = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.eos_token_id,
        "use_cache": False,  # lower peak memory for Colab CPU execution
    }
    if do_sample:
        generation_options.update({
            "temperature": temperature,
            "top_p": top_p,
        })

    torch.manual_seed(seed)
    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            **generation_options,
        )

    input_length = model_inputs["input_ids"].shape[1]
    new_token_ids = output_ids[0, input_length:]
    return tokenizer.decode(new_token_ids, skip_special_tokens=True).strip()


def next_token_candidates(text, top_k=10):
    model_inputs = tokenizer(text, return_tensors="pt")
    model_inputs = {name: value.to(device) for name, value in model_inputs.items()}
    with torch.inference_mode():
        logits = model(**model_inputs, use_cache=False).logits
    probabilities = torch.softmax(logits[0, -1].float(), dim=-1)
    top_probabilities, top_ids = torch.topk(probabilities, top_k)
    return pd.DataFrame({
        "token_id": top_ids.cpu().numpy(),
        "token_piece": [repr(tokenizer.decode([token_id])) for token_id in top_ids],
        "probability": top_probabilities.cpu().numpy(),
    })

## Question 1 — How does text become tokens?

For the three provided English, Chinese and mixed-language sentences:

1. calculate the number of characters and tokens;
2. display the results as a table;
3. choose one sentence and display each token position, token ID and decoded token piece;
4. explain why the number of tokens is not necessarily the number of words or characters.

In [ ]:
sentences = [
    "Traffic congestion affects travel time.",
    "香港的交通流量正在增加。",
    "We study AI for transport applications.",
]

# TODO: build and display a sentence-level token-count table.
# TODO: build a token-level table for one sentence.

## Question 2 — How does context change the next token?

Use `next_token_candidates()` to inspect the ten most likely next tokens for:

```text
Traffic congestion can be reduced by
A bridge inspection should check
```

Display both tables and draw two horizontal probability bar charts. Describe one clear difference between the candidate distributions.

In [ ]:
prompts = [
    "Traffic congestion can be reduced by",
    "A bridge inspection should check",
]

# TODO: calculate the top ten candidates for both prompts.
# TODO: draw two readable horizontal bar charts.

## Question 3 — How do generation temperature and top-p change sampling?

Here `temperature` rescales the model's **final next-token logits** before a token is sampled. It is a generation setting, not the $1/\sqrt{d_k}$ scaling inside attention. `top_p` then sorts token probabilities and retains the smallest candidate set whose cumulative probability reaches $p$; probabilities outside that set are removed and the retained probabilities are renormalised.

For the prompt below, generate four short responses using every combination of:

```python
temperatures = [0.4, 0.8]
top_ps = [0.8, 0.95]
```

Use `do_sample=True`, which activates random sampling instead of greedy argmax. Use `seed=42` to make each setting reproducible, and keep the same `max_new_tokens` (the maximum number of newly generated tokens, not words) for every run. Put the settings and outputs in a table. Which settings appear more conservative, and which show more variation?

In [ ]:
prompt = "Suggest one simple way to improve pedestrian safety near a school."
temperatures = [0.4, 0.8]
top_ps = [0.8, 0.95]

# TODO: run the four combinations and display the outputs in a table.

## Question 4 — Extension: can a clearer prompt change the format?

Use the inspection note below:

```text
Beam B12 has moderate corrosion near the support. Schedule a detailed inspection within one month.
```

Generate one response with a vague instruction such as “summarise this note”. Then generate another response that explicitly requests JSON with the keys `component`, `condition`, `location` and `action`. Compare the outputs.

Explain why this changes the input context but does not change the model parameters.

In [ ]:
inspection_note = (
    "Beam B12 has moderate corrosion near the support. "
    "Schedule a detailed inspection within one month."
)

# TODO: generate one response with a vague prompt.
# TODO: generate one response with an explicit JSON-format prompt.
# TODO: print both responses clearly.

## Submission checklist

- Token tables use readable column names.
- Probability plots label the token pieces and probabilities.
- The decoding experiment changes only the requested parameters.
- Explanations distinguish prompt/context changes from parameter training.
- No API key, training loop or external dataset is required.